In [ ]:
# Работа с данными
import pandas as pd
import numpy as np
import datetime
import itertools
# import category_encoders as ce

# Общие функции
import sys
sys.path.append("/home/jovyan/share/1libs/Functions/") # go to parent dir
from oskar_ml import *

# Визуализация
import matplotlib.pyplot as plt
from copy import deepcopy
import seaborn as sns
from scipy import stats


# Игнорирование системных предупреждений
import warnings
warnings.filterwarnings('ignore')

# Манипуляция данными
import gc #сбор мусора
gc.enable()

# Визуализация процесса расчёта
from tqdm import tqdm, tqdm_notebook
tqdm.pandas()

# Отобразить все колонки
from IPython.display import display
pd.set_option('display.max_columns', None)
# Отобразить 230 строк
pd.set_option("display.max_rows", 230)

# Модели
from catboost import CatBoostRegressor, Pool, EShapCalcType, EFeaturesSelectionAlgorithm, sum_models
from sklearn.datasets import make_regression
from sklearn.model_selection import train_test_split

# Разделение данных
from sklearn.model_selection import train_test_split, TimeSeriesSplit
from sklearn.model_selection import cross_val_score, GridSearchCV, KFold
from sklearn.model_selection import StratifiedKFold,  RandomizedSearchCV

# Метрики
from sklearn.metrics import mean_squared_error, mean_absolute_error
from sklearn import metrics

# Оптимизация гиперпараметров
import optuna

In [ ]:
# with open(os.path.join( "spr_minimization.pickle"), "rb") as f:
#     group = pickle.load(f)
#     all_groups.update({group_name: group})

In [ ]:
df = pd.read_parquet('/home/jovyan/old_home/Litigant/data/processed/df_final_2.parquet')
df.shape

In [ ]:
filials = ['Архангельский', 
        'Владимирский', 
        'Кемеровский', 
        'Курский', 
        'Магнитогорский', 
        'Марийский', 
        'Мурманский', 
        'Омский', 
        'Пермский', 
        'Петропавловск-Камчатский', 
        'Уфимский', 
        'Ярославский']

In [ ]:
df = df[df['FILIAL'].isin(filials)]
df.shape

In [ ]:
df = df.rename({'EMRValue': 'EMR_VALUE'}, axis=1)

In [ ]:
ftrs = [
    'INCIDENT_NUMBER',
    'LOSS_NUMBER',
    'FILIAL',
    'EVENT_CREATED_BY_GIBDD_FLAG',
    'VICTIM_MAX_WEIGHT',
    'RECIEVE_METHOD',
    'APPLICANT_FORM',
    'APPLICANT_AGE',
    'VICTIM_VEHICLE_CATEGORY',
    'GUILTY_CAPACITY_ENGINE',
    'VICTIM_VEHICLE_AGE',
    'EVENT_DATE',
    'PAYMENT_ORDER_DATE_TIME',
    'AMOUNT_REPAIR',
    'LOSS_UNIT_ZONE',
    'VICTIM_VEHICLE_COUNTRY']

In [ ]:
df_test = df[ftrs][7:8]

In [ ]:
for col in [col for col in df_test.columns if 'DATE' in col]:
    df_test[col] = df_test[col].astype(str)

In [ ]:
import json
import requests

header = {'Content-Type': 'application/json;charset=utf-8', \
                  'Accept': 'application/json'}

In [ ]:
import pandas as pd
import numpy as np
from typing import Dict, Set

def _get_type_with_bits(val) -> str:
    if isinstance(val, np.generic):
        return val.dtype.name
    if isinstance(val, bool):
        return 'bool8'
    if isinstance(val, float):
        return 'float64'
    if isinstance(val, int):
        bits = val.bit_length() or 8
        return f'int{bits}'
    if isinstance(val, str):
        return 'str'
    return type(val).__name__

def get_column_type_mapping(df: pd.DataFrame) -> Dict[str, Set[str]]:
    if not isinstance(df, pd.DataFrame):
        raise TypeError(f"Ожидается pd.DataFrame, получено {type(df).__name__}")
    if len(df.columns) == 0:
        raise ValueError("DataFrame не содержит колонок. Проверьте загрузку данных.")

    mapping = {}
    for col in df.columns:
        non_null = df[col].dropna()
        has_nan = df[col].isna().any()

        if non_null.empty:
            mapping[col] = {'empty'}
            continue

        types = {_get_type_with_bits(val) for val in non_null.unique()}
        if has_nan:
            types.add('nan')
        mapping[col] = types

    return mapping

In [ ]:
result = get_column_type_mapping(df[ftrs])

In [ ]:
df_test['IS_LAWYER'] = 1

In [ ]:
data_to_send = {}
data_to_send['main_model'] = 'querulus_ansamble_2026_22_04_v1'
data_to_send['main_request'] = df_test.to_dict(orient='records')#.to_json(date_format='iso', force_ascii = False, orient='records')
# json.dumps(data_to_send, ensure_ascii=False)

# data_to_send['second_model'] = 'DVP_OSAGO_2024_09_24_13_48_51'
# data_to_send['second_request'] = df_[:5].to_dict(orient='records')#.to_json(date_format='iso', force_ascii = False, orient='records')
json.dumps(data_to_send, ensure_ascii=False)

In [ ]:
data_to_send = {}
data_to_send['main_model'] = 'querulus_ansamble_2026_04_18_v1'
data_to_send['main_request'] = df_test.to_dict(orient='records')#.to_json(date_format='iso', force_ascii = False, orient='records')
# json.dumps(data_to_send, ensure_ascii=False)

# data_to_send['second_model'] = 'DVP_OSAGO_2024_09_24_13_48_51'
# data_to_send['second_request'] = df_[:5].to_dict(orient='records')#.to_json(date_format='iso', force_ascii = False, orient='records')
json.dumps(data_to_send, ensure_ascii=False)

In [ ]:
data_to_send_ = json.dumps(data_to_send, ensure_ascii=False) 

resp = requests.post("http://0.0.0.0:8080/api/predict", \
                    data = data_to_send_.encode('utf-8'),\
                    headers = header)
# if 'predictions' in resp.json().keys():
#     print('ok')
#     answer = eval(resp.json()['predictions'])
resp.json()

In [ ]:
_df_resp = pd.read_excel('/home/jovyan/old_home/Litigant/integration/prepared_dfs/prepared_df_20260416_171032_548921.xlsx')
_df_resp = _df_resp.drop('Unnamed: 0', axis=1)
_df_resp.shape

In [ ]:
_df_orig = pd.read_parquet('/home/jovyan/old_home/Litigant/data/processed/df_for_service_test.parquet')
_df_orig = _df_orig[['INCIDENT_NUMBER', 'LOSS_NUMBER', 'FILIAL',
                     'EVENT_CREATED_BY_GIBDD_FLAG', 'VICTIM_MAX_WEIGHT', 'RECIEVE_METHOD',
                     'APPLICANT_FORM', 'APPLICANT_AGE', 'VICTIM_VEHICLE_CATEGORY',
                     'GUILTY_CAPACITY_ENGINE', 'VICTIM_VEHICLE_AGE', 'EVENT_DATE',
                     'PAYMENT_ORDER_DATE_TIME', 'AMOUNT_REPAIR', 'LOSS_UNIT_ZONE',
                     'VICTIM_VEHICLE_COUNTRY', 'EVENT_YEAR', 'APPLY_DELAY']]
_df_orig.shape

In [ ]:
_df_orig['INCIDENT_NUMBER'] = _df_orig['INCIDENT_NUMBER'].astype(int)
_df_orig['LOSS_NUMBER'] = _df_orig['LOSS_NUMBER'].astype(int)
_df_orig['AMOUNT_REPAIR'] = _df_orig['AMOUNT_REPAIR'].astype(int)
_df_orig['EVENT_CREATED_BY_GIBDD_FLAG'] = _df_orig['EVENT_CREATED_BY_GIBDD_FLAG'].astype(int)
_df_resp['EVENT_DATE'] = _df_resp['EVENT_DATE'].astype('datetime64[ns]')
_df_resp['PAYMENT_ORDER_DATE_TIME'] = _df_resp['PAYMENT_ORDER_DATE_TIME'].astype('datetime64[ns]')
_df_resp['EVENT_YEAR'] = _df_resp['EVENT_YEAR'].astype('int32')

In [ ]:
for col in _df_orig.columns:
    print(col, _df_orig[col].dtypes, _df_resp[col].dtypes)

In [ ]:
display(_df_orig.sample(5, random_state=0))
display(_df_resp.sample(5, random_state=0))

In [ ]:
display(_df_orig.head(5))
display(_df_resp.head(5))

In [ ]:
data_to_send_ = json.dumps(data_to_send, ensure_ascii=False) 

resp = requests.post("http://perf-pydsp-test1.vsk.ru:10637/api/predict", \
                    data = data_to_send_.encode('utf-8'),\
                    headers = header)
# if 'predictions' in resp.json().keys():
#     print('ok')
#     answer = eval(resp.json()['predictions'])
resp.json()

In [ ]:
data_to_send_ = json.dumps(data_to_send, ensure_ascii=False) 

resp = requests.post("http://perf-pydsp-cls.vsk.ru:10537/api/predict", \
                    data = data_to_send_.encode('utf-8'),\
                    headers = header)
# if 'predictions' in resp.json().keys():
#     print('ok')
#     answer = eval(resp.json()['predictions'])
resp.json()